# Mutual Fund Recommender — LangChain edition (Colab runner)

All the logic lives in the `backend/mf_recommender` package; this notebook installs it, finds your files and serves the API for the React app.

**Pipeline:** goal → (Phi-3-mini query rewrite) → hybrid retrieval (FAISS + BM25, fused with reciprocal rank fusion, filtered by category & risk) → peer-relative ranking → live NAV → grounded explanations (Phi-3-mini + LoRA, with a fact-checked fallback).

**Setup (no paths to edit):**
1. Unzip the project on your computer and upload the `mutual_funds_recommendation_system` folder anywhere in My Drive (e.g. `My Drive/Colab_Notebooks/`).
2. Runtime → Change runtime type → **T4 GPU**.
3. Store your ngrok token in Colab Secrets (🔑 icon) as `NGROK_AUTHTOKEN` (only needed for the API step).
4. Runtime → Run all.

**First run:** if no LoRA adapter for Phi-3-mini exists yet, the notebook fine-tunes one (about 1–1.5 hours on a T4) before loading the model. Later runs reuse it and start in about 5 minutes. Everything it creates (search index, adapter) is saved in the project's `artifacts/` folder in Drive.

In [ ]:
import os, sys, glob
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'

def find_in_drive(relative_path, max_depth=4):
    """Paths in My Drive ending in `relative_path`, up to `max_depth` folders deep, newest first."""
    hits = []
    for depth in range(0, max_depth + 1):
        hits += glob.glob(os.path.join(DRIVE, *(['*'] * depth), relative_path))
    return sorted(set(hits), key=os.path.getmtime, reverse=True)

# 1) The project folder: wherever you put it, whatever the folders are called.
MARKER = os.path.join('backend', 'mf_recommender', 'service.py')
projects = [h[: -len(MARKER) - 1] for h in find_in_drive(MARKER)]
if not projects:
    raise FileNotFoundError('Project not found in My Drive: upload the unzipped mutual_funds_recommendation_system folder')
PROJECT_DIR = projects[0]  # newest upload wins
if len(projects) > 1:
    print('Several copies found; using the newest. Older copies (safe to delete):', projects[1:])
print('Project:', PROJECT_DIR)

BACKEND_DIR = os.path.join(PROJECT_DIR, 'backend')
os.chdir(BACKEND_DIR)
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)  # so `import mf_recommender` works in this notebook

# 2) Catch an incomplete upload early.
EXPECTED = ['backend/mf_recommender/retrieval.py', 'backend/tests/conftest.py',
            'backend/scripts/train_lora.py', 'backend/pytest.ini']
missing = [f for f in EXPECTED if not os.path.exists(os.path.join(PROJECT_DIR, f))]
if missing:
    print('WARNING: missing from your Drive copy:', missing, '- re-upload the full folder from the zip')

# 3) The fund data: the project's data/ folder, otherwise the newest copy anywhere in My Drive,
#    otherwise ask you to pick the file (it is then saved into the project for next time).
DATA_CSV = os.path.join(PROJECT_DIR, 'data', 'updated_data_with_nav.csv')
if not os.path.isfile(DATA_CSV):
    elsewhere = find_in_drive('updated_data_with_nav.csv')
    if elsewhere:
        DATA_CSV = elsewhere[0]
        os.environ['MF_DATA_PATH'] = DATA_CSV
    else:
        if find_in_drive('updated_data_with_nav.gsheet'):
            print('Google Drive converted updated_data_with_nav.csv into a Google Sheet when it was uploaded,\n'
                  'so it can no longer be read as a CSV. (Drive Settings > "Convert uploads" causes this.)')
        print('Please pick updated_data_with_nav.csv from the unzipped data/ folder on your computer:')
        from google.colab import files
        uploaded = files.upload()
        name = next((n for n in uploaded if n.lower().endswith('.csv')), None)
        if name is None:
            raise FileNotFoundError('No CSV selected; pick updated_data_with_nav.csv and run this cell again')
        os.makedirs(os.path.dirname(DATA_CSV), exist_ok=True)
        with open(DATA_CSV, 'wb') as f:  # written from Colab, so Drive keeps it as a real CSV
            f.write(uploaded[name])
        print('Saved; it will be found automatically next time.')
print('Fund data:', DATA_CSV)

In [ ]:
# Installs everything directly (doesn't depend on the requirements files).
!pip install -q "langchain-core>=1.0,<2" "langchain-community>=0.4,<0.5" "langchain-huggingface>=1.0,<2" \
    faiss-cpu rank-bm25 sentence-transformers fastapi "uvicorn[standard]" \
    transformers peft accelerate datasets pyngrok pytest httpx

# Colab ships an old torchao (0.10). Recent peft refuses to set up LoRA while it is installed
# ('Found an incompatible version of torchao'). This project doesn't use torchao (it is for
# quantization), so remove it.
!pip uninstall -y -q torchao

## Configure
The LLM is Phi-3-mini (`BASE_MODEL`). Set `MF_LLM_BACKEND = 'none'` for a fast, template-only run without loading it. Other settings: `backend/.env.example`.

In [ ]:
from mf_recommender.config import ARTIFACTS_DIR, find_adapter
from mf_recommender.model_utils import adapter_fits

BASE_MODEL = 'microsoft/Phi-3-mini-4k-instruct'   # the LLM; adapters are matched to it
os.environ['MF_BASE_MODEL'] = BASE_MODEL
os.environ['MF_LLM_BACKEND'] = 'hf'               # 'none' = fast template-only run, no LLM
os.environ['MF_LIVE_NAV'] = 'true'

def adapters_in_drive(base_model):
    """LoRA adapters trained on `base_model` anywhere in My Drive (training checkpoints skipped)."""
    return [os.path.dirname(c) for c in find_in_drive('adapter_config.json')
            if 'checkpoint' not in c and adapter_fits(os.path.dirname(c), base_model)]

def pick_adapter():
    # The project's artifacts/ first (where train_lora.py saves), then anywhere else in Drive.
    return find_adapter(BASE_MODEL) or next(iter(adapters_in_drive(BASE_MODEL)), None)

adapter = pick_adapter()
if adapter:
    os.environ['MF_ADAPTER_PATH'] = adapter
    print('Using LoRA adapter:', adapter)
else:
    os.environ.pop('MF_ADAPTER_PATH', None)
    print(f'No LoRA adapter for {BASE_MODEL} yet; the fine-tuning cell below will train one')
print('Index and artifacts folder:', ARTIFACTS_DIR)

## Check and build the index
Tests run offline (no model downloads) in a few seconds.

In [ ]:
!python -m pytest -q   # expect: 93 passed

In [ ]:
!python -m mf_recommender.build_index

## Fine-tune Phi-3-mini with LoRA
`scripts/train_lora.py` builds about 1,700 training examples from the fund data, writes each in Phi-3-mini's chat format, and trains LoRA on its attention layers (`qkv_proj`, `o_proj`) with the loss on the answer only. About 1–1.5 hours on a T4; the adapter is saved to `artifacts/lora-phi-3-mini-4k-instruct` and used automatically from then on.

This cell runs **before** the model is loaded for serving: a T4 can't hold a second copy of Phi-3-mini while training. `RETRAIN = 'auto'` trains only when no adapter exists yet; set `True` to retrain, `False` to skip. After a retrain with `True`, set it back to `'auto'`, or every *Run all* will train again.

**While it trains:** the log should start with `Precision: float16 on Tesla T4`, and the progress bar should show well under a minute per step. Keep this tab open: free Colab can disconnect an idle session, and an interrupted run saves nothing.

In [ ]:
RETRAIN = 'auto'   # 'auto' | True | False

if RETRAIN is True or (RETRAIN == 'auto' and not os.environ.get('MF_ADAPTER_PATH')):
    # An older train_lora.py picked bfloat16 on a T4, which the T4 only emulates: ~4 hours instead of ~1.
    if 'bf16 = cuda and torch.cuda.is_bf16_supported()' in open('scripts/train_lora.py').read():
        raise RuntimeError('backend/scripts/train_lora.py in your Drive is an older version that trains about 8x '
                           'slower on a T4. Replace it with the latest one, then run this cell again.')
    if 'service' in globals():          # free the GPU if the model is already loaded
        import gc, torch
        del service; gc.collect(); torch.cuda.empty_cache()
    !python scripts/train_lora.py
    adapter = find_adapter(BASE_MODEL)
    if not adapter:
        # Stop here rather than serve the untuned model by accident.
        raise RuntimeError('Fine-tuning did not save an adapter: scroll up for the error (often CUDA out of '
                           'memory: Runtime > Restart session, then run all again). Without an adapter, '
                           'Phi-3-mini runs untuned and most explanations fall back to the data-based text.')
    os.environ['MF_ADAPTER_PATH'] = adapter
    print('Using new LoRA adapter:', adapter)
elif os.environ.get('MF_ADAPTER_PATH'):
    print('Skipping fine-tuning; using adapter:', os.environ['MF_ADAPTER_PATH'])
else:
    print('WARNING: no fine-tuned adapter, so Phi-3-mini will run untuned. Set RETRAIN = True to train one.')

## Try it in Python

In [ ]:
import logging
import pandas as pd
from mf_recommender.config import Settings
from mf_recommender.llm import describe_model
from mf_recommender.service import RecommenderService

# Show the project's log lines, e.g. why a model explanation was rejected by the quality gate.
logging.basicConfig(format='%(levelname)s %(name)s: %(message)s')
logging.getLogger('mf_recommender').setLevel(logging.INFO)

service = RecommenderService.from_settings(Settings.from_env())  # loads embeddings + Phi-3-mini once
print('Serving:', describe_model(service.settings, service.llm is not None))  # fine_tuned should be True

result = service.recommend(
    'I want long-term growth over 10 years with low costs',
    category=None,   # None = let the goal decide; or e.g. 'Large Cap Mutual Funds'
    risk='High',
)
print('Rewritten query:', result.query['rewritten'])
print('Notice:', result.notice)
display(pd.DataFrame(result.funds)[['scheme_name', 'category', 'risk_level', 'NAV', 'returns_1yr',
                                    'returns_3yr', 'returns_5yr', 'expense_ratio', 'score']])
for f in result.funds:
    print(f"\n{f['scheme_name']} [{f['explanation_source']}]\n  {f['explanation']}")

## Serve the API through ngrok
Reuses the `service` loaded above, so Phi-3-mini isn't loaded twice. Put the printed URL in `frontend/.env` as `REACT_APP_API_URL`.

In [ ]:
import threading
import time
import requests
import uvicorn
from google.colab import userdata
from pyngrok import conf, ngrok
from mf_recommender.api import create_app

app = create_app(service=service)
server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info'))
threading.Thread(target=server.run, daemon=True).start()

conf.get_default().auth_token = userdata.get('NGROK_AUTHTOKEN')
public_url = ngrok.connect(8000).public_url
print('REACT_APP_API_URL =', public_url)
for _ in range(15):   # the server thread needs a moment to start
    try:
        print('Health:', requests.get(f'{public_url}/health', headers={'ngrok-skip-browser-warning': 'true'},
                                      timeout=10).json())
        break
    except (requests.RequestException, ValueError):
        time.sleep(1)

In [ ]:
import requests
r = requests.post(f'{public_url}/recommend',
                  json={'goal': 'Park my emergency fund safely for 3 months', 'category': None, 'risk': 'Low'},
                  headers={'ngrok-skip-browser-warning': 'true'}, timeout=300)
r.raise_for_status()
[(f['scheme_name'], f['category']) for f in r.json()['funds']]

In [ ]:
# Stop the API: set STOP = True and run this cell only when you're finished.
# (Guarded so 'Run all' leaves the server running.)
STOP = False
if STOP:
    server.should_exit = True
    ngrok.kill()
    print('API stopped')